# AI Agent: Hello World

Welcome to your AI Agent sandbox! This notebook shows you how to build a **basic, provider-agnostic AI agent**.

Instead of locking our code to just one AI (like OpenAI), we use a base adapter (`LLMAdapter`). This lets us easily switch between different AIs (like Gemini, Claude, or Copilot) without changing the core agent code.

**Run the cells top to bottom, one at a time.**

### 1. Environment Setup
First, we install the necessary packages for the AI providers and for loading environment variables.

In [ ]:
# Run this once, then comment it out
!pip install -q anthropic openai google-genai python-dotenv
print("✅ Packages ready")

### 2. Authentication
Load your API keys securely. It is best to keep these in a `.env` file in the same folder as this notebook.

In [1]:
import os
from dotenv import load_dotenv

# Load from .env file if you have one
load_dotenv()

key = os.getenv('GEMINI_API_KEY', '')
if key:
    print(f'✅ API key loaded: {key[:10]}...')
else:
    print('❌ API key NOT found — please set it in your .env file')

✅ API key loaded: AIzaSyCd0P...


### 3. Core Structures & Annotations
Here we define the standard building blocks for our agent. 

**What is `@dataclass`?** 
It is a Python shortcut. Instead of writing long setup code (`def __init__(self, name...):`), putting `@dataclass` above a class automatically builds all that background code for you. It is perfect for classes that just hold data.

**What are Annotations (Type Hints)?** 
When you see `: str` or `: dict`, it tells Python (and your code editor) exactly what type of data belongs there. 
- `name: str` means "name must be text". 
- `-> AgentResponse` means "this function will return an AgentResponse object".
This helps prevent bugs when talking to the AI.

**What are `@abstractmethod`s?** 
In the `LLMAdapter` class, we create empty functions marked with `@abstractmethod`. This acts as a strict blueprint. It means: *"Any AI we plug in later MUST have its own version of these specific functions."*

In [2]:
import json
from dataclasses import dataclass
from abc import ABC, abstractmethod
from typing import Any

# A blueprint for the tools our AI is allowed to use
@dataclass
class ToolDefinition:
    name: str           # The name of the tool (e.g., 'get_attendance')
    description: str    # What the tool does
    parameters: dict    # The inputs the tool needs (e.g., 'student_id')

# When the AI asks to use a tool, it sends this data structure back to us
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict

# The standard format we expect back from ANY AI model
@dataclass
class AgentResponse:
    is_final: bool              # True if AI gave an answer, False if it wants to use a tool
    final_text: str | None      # The actual text answer (if is_final is True)
    tool_calls: list[ToolCall]  # List of tools the AI wants to use (if is_final is False)

# The base Blueprint that all AIs (Gemini, Claude, etc.) must follow
class LLMAdapter(ABC):
    
    @abstractmethod
    def build_tool_definitions(self, tools: list[ToolDefinition]) -> list[dict]: 
        """Converts our generic ToolDefinitions into the specific format the AI API expects."""
        ...
        
    @abstractmethod
    def call(self, system_prompt: str, messages: list, tools: list) -> AgentResponse: 
        """Sends the conversation to the AI API and returns a standard AgentResponse."""
        ...
        
    @abstractmethod
    def build_tool_result_message(self, tool_call: ToolCall, result: str, assistant_message: list) -> list[dict]: 
        """Packages the data we fetched from our Python code so the AI can read it."""
        ...

print('✅ Core structures defined')

✅ Core structures defined


### 4. Implementing the Gemini Adapter
Below is the code that connects our agent specifically to **Gemini**. It takes the blueprint from `LLMAdapter` and fills in the actual code needed to talk to Google's servers.

In [3]:
# ── GEMINI ADAPTER ──
class GeminiAdapter(LLMAdapter):
    """Google Gemini via the new google-genai SDK."""

    # Setup the connection to Google
    def __init__(self, model: str = "gemini-2.5-flash"):
        from google import genai
        from google.genai import types
        self.client = genai.Client() # Automatically uses GEMINI_API_KEY from env variables
        self.types = types
        self.model_name = model

    # Method 1: Format tools for Gemini
    def build_tool_definitions(self, tools: list[ToolDefinition]) -> list[dict]:
        declarations = []
        for tool in tools:
            declarations.append(
                self.types.FunctionDeclaration(
                    name=tool.name,
                    description=tool.description,
                    parameters={
                        "type": "object",
                        "properties": {
                            k: {"type": "string", "description": v.get("description", "")}
                            for k, v in tool.parameters.get("properties", {}).items()
                        },
                        "required": tool.parameters.get("required", []),
                    },
                )
            )
        return [self.types.Tool(function_declarations=declarations)]

    # Method 2: Send the prompt and get a response
    def call(self, system_prompt: str, messages: list, tools: list) -> AgentResponse:
        history = []
        for m in messages[:-1]:
            role = "user" if m["role"] == "user" else "model"
            content = m["content"] if isinstance(m["content"], str) else str(m["content"])
            history.append(self.types.Content(role=role, parts=[self.types.Part.from_text(text=content)]))

        config = self.types.GenerateContentConfig(
            system_instruction=system_prompt,
            tools=tools,
        )

        # Format the newest message
        last_content = messages[-1]["content"]
        if isinstance(last_content, list):
            parts = []
            for item in last_content:
                if isinstance(item, dict) and item.get("type") == "tool_result":
                    parts.append(
                        self.types.Part.from_function_response(
                            name=item.get("name", "tool"),
                            response={"result": item.get("content", "")},
                        )
                    )
            if not parts:
                parts = [self.types.Part.from_text(text=str(last_content))]
            last_msg = self.types.Content(role="user", parts=parts)
        else:
            last_msg = self.types.Content(role="user", parts=[self.types.Part.from_text(text=last_content)])

        # Call the Google API
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=history + [last_msg],
            config=config,
        )

        # Check if Gemini wants to use a tool
        fn_calls = []
        if response.candidates and response.candidates[0].content and response.candidates[0].content.parts:
            for part in response.candidates[0].content.parts:
                if part.function_call:
                    fc = part.function_call
                    call = ToolCall(id=fc.name, name=fc.name, arguments=dict(fc.args))
                    fn_calls.append(call)

        # Return the final text if no tools were called
        if not fn_calls:
            text = response.text if hasattr(response, "text") else str(response)
            return AgentResponse(is_final=True, final_text=text, tool_calls=[])

        # Return the tool requests if it needs more data
        return AgentResponse(is_final=False, final_text=None, tool_calls=fn_calls)

    # Method 3: Format the fetched data back to Gemini
    def build_tool_result_message(self, tool_call: ToolCall, result: str, assistant_message: list) -> list[dict]:
        return [{
            "role": "user",
            "content": [{
                "type": "tool_result",
                "name": tool_call.name,
                "content": result,
            }],
        }]

print('✅ Gemini Adapter ready')

✅ Gemini Adapter ready


### 5. The Agent Loop
This is the brain of our agent. It uses a **`While True`** loop to figure out what to do.

Here is how the `run()` method works:
1. **Send Prompt:** It gives your question to the AI.
2. **Check Response:** If the AI has enough information, it answers and the loop breaks (`if response.is_final`).
3. **Execute Tool:** If the AI needs data, it asks to use a tool. We pause, run the Python function (`self.tool_runner`), and give the data back to the AI. The loop repeats.

In [4]:
class Agent:
    # __init__ runs once when we first create the agent
    def __init__(self, adapter: LLMAdapter, tools: list, tool_runner, system_prompt: str, verbose: bool = True):
        self.adapter = adapter
        self.tools = tools
        self.tool_runner = tool_runner
        self.system_prompt = system_prompt
        self.verbose = verbose
        self._tool_defs = adapter.build_tool_definitions(tools)

    # run() is the continuous ReAct (Reason + Act) loop
    def run(self, question: str) -> str:
        messages = [{'role': 'user', 'content': question}]
        if self.verbose:
            print(f"\n{'='*55}")
            print(f"  Question : {question}")
            print(f"  Provider : {self.adapter.__class__.__name__}")
            print(f"{'='*55}")

        while True:
            # 1. Ask the AI
            response = self.adapter.call(
                system_prompt=self.system_prompt,
                messages=messages,
                tools=self._tool_defs,
            )
            
            # 2. Break condition: The LLM gives a final text answer
            if response.is_final:
                if self.verbose:
                    print(f"\n  ✅ Answer: {response.final_text}")
                return response.final_text

            # 3. Action condition: The LLM wants to use a tool to get more data
            for tool_call in response.tool_calls:
                if self.verbose:
                    print(f"\n  🔧 Tool  : {tool_call.name}({tool_call.arguments})")
                
                # 4. Execute the actual Python function (the tool_runner)
                result = self.tool_runner(tool_call.name, tool_call.arguments)
                
                if self.verbose:
                    print(f"  📦 Result: {result}")
                
                # 5. Pass the new data back into the message history so the loop can try again
                new_messages = self.adapter.build_tool_result_message(
                    tool_call=tool_call, result=result, assistant_message=messages
                )
                messages.extend(new_messages)

print('✅ Agent loop ready')

✅ Agent loop ready


### 6. Tools & System Prompt
This is where you give the agent its abilities.

#### What are Tools?
AIs can only predict text; they can't run code. The `TOOLS` list acts as a menu we show the AI so it knows what it is allowed to ask us to do.

#### What is the `tool_runner` Method?
This is the "Router". When the AI generates text saying "use get_attendance", the `tool_runner` intercepts that string, matches it to the real Python function `get_attendance()`, runs the code, and returns the raw data back.

#### What is a System Prompt?
The `SYSTEM_PROMPT` is the agent's persona. It tells the AI who it is and what rules to follow.(e.g., \"always use tools to fetch real data\").\n",

In [8]:
# 1. The Database (Using a simple dictionary here, but could be Snowflake or Redshift)
STUDENTS = {
    'S001': {'name': 'Aarav Sharma',  'attendance_pct': 91.5, 'fees_paid': True,  'fees_due': 0},
    'S002': {'name': 'Priya Mehta',   'attendance_pct': 63.2, 'fees_paid': False, 'fees_due': 18500},
    'S003': {'name': 'Rohan Verma',   'attendance_pct': 78.0, 'fees_paid': True,  'fees_due': 0},
    'S004': {'name': 'Sneha Iyer',    'attendance_pct': 54.1, 'fees_paid': False, 'fees_due': 22000},
}

# 2. The Python Functions (The code that fetches the data)
def get_attendance(student_id: str) -> dict:
    s = STUDENTS.get(student_id)
    if not s: return {'error': f'Student {student_id} not found'}
    return {'student_id': student_id, 'name': s['name'],
            'attendance_pct': s['attendance_pct'],
            'status': 'OK' if s['attendance_pct'] >= 75 else 'LOW — below 75%'}

def get_fee_status(student_id: str) -> dict:
    s = STUDENTS.get(student_id)
    if not s: return {'error': f'Student {student_id} not found'}
    return {'student_id': student_id, 'name': s['name'],
            'fees_paid': s['fees_paid'], 'amount_due': s['fees_due'],
            'status': 'Cleared' if s['fees_paid'] else f"PENDING ₹{s['fees_due']:,}"}

def list_all_students() -> dict:
    return {'students': [{'id': k, 'name': v['name']} for k, v in STUDENTS.items()]}

# 3. The Tool Runner (The router that connects the AI's requests to the real functions)
def tool_runner(name: str, args: dict) -> str:
    # Map the string name to the actual Python function
    fns = {
        'get_attendance':    get_attendance,
        'get_fee_status':    get_fee_status,
        'list_all_students': list_all_students,
    }
    fn = fns.get(name)
    
    # If the AI asked for a tool that doesn't exist, tell it.
    if not fn: return json.dumps({'error': f'Unknown tool: {name}'})
    
    # Run the function with the arguments the AI provided, and format output to JSON
    return json.dumps(fn(**args))

# 4. The Tool Definitions (The "menu" of abilities for the AI)
TOOLS = [
    ToolDefinition(
        name='get_attendance',
        description='Get attendance % for a student. Use for attendance/absence questions.',
        parameters={'type': 'object', 'properties': {'student_id': {'type': 'string', 'description': 'e.g. S001'}}, 'required': ['student_id']},
    ),
    ToolDefinition(
        name='get_fee_status',
        description='Get fee payment status. Use for fee/dues/payment questions.',
        parameters={'type': 'object', 'properties': {'student_id': {'type': 'string', 'description': 'e.g. S001'}}, 'required': ['student_id']},
    ),
    ToolDefinition(
        name='list_all_students',
        description='List all students. Use when no specific student is mentioned.',
        parameters={'type': 'object', 'properties': {}},
    ),
]

# 5. The System Prompt (The Agent's Persona)
SYSTEM_PROMPT = (
    'You are Hello_World_agent, an AI assistant for school management. '
    'Always use tools to fetch real data before answering. '
    'Give concise, actionable answers for a school principal.'
    'Give count of how many time you used the tools to get the answer of previous question.'
)
print('✅ Tools and data ready')

✅ Tools and data ready


### 7. Execution
Initialize the agent with our chosen adapter (Gemini) and test it out. Notice how it autonomously decides to use the `get_attendance` tool to find the answer!

In [ ]:
# To switch models later, simply change this one line (e.g., adapter = ClaudeAdapter())
adapter = GeminiAdapter()     

agent = Agent(
    adapter=adapter,
    tools=TOOLS,
    tool_runner=tool_runner,
    system_prompt=SYSTEM_PROMPT,
    verbose=True,
)
print(f'✅ Agent initialized successfully with {adapter.model_name}\n')

agent.run('What is the attendance status of student S002?')

✅ Agent initialized successfully with gemini-2.5-flash


  Question : i m feeling tired today
  Provider : GeminiAdapter


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 13.42342702s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 